In [2]:
import pandas as pd
import json

In [3]:
import pandas as pd
import json
from pathlib import Path
import os

base_dir = Path("finetuning_eval")
json_files = list(base_dir.rglob("metrics.json"))
print(json_files)
all_dfs = []

for json_file in json_files:
    with open(json_file, "r") as f:
        data = json.load(f)

    scores_list = []
    for entry in data.get("all_primary_scores", []):
        alias, score = entry.rsplit(": ", 1)
        scores_list.append({"alias": alias, "primary_score": float(score)})
    df_primary_scores = pd.DataFrame(scores_list)

    tasks_list = []
    for task in data.get("tasks", []):
        entry = {"alias": task["alias"], "num_instances": task.get("num_instances"), "task_suite": json_file.parts[1]}
        entry.update(task.get("metrics", {}))
        tasks_list.append(entry)
    df_tasks = pd.DataFrame(tasks_list)

    df = pd.merge(df_primary_scores, df_tasks, on="alias", how="left")
    df["model"] = os.path.basename(json_file.parent)
    all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)

def rename(name):
    if "arc_easy" in name:
        return "ARC\_E"
    elif "arc_challenge" in name:
        return "ARC\_C"
    elif "boolq" in name:
        return "BoolQ"
    elif "csqa" in name:
        return "CSQA"
    elif "hellaswag" in name:
        return "HSwag"
    elif "mmlu_pro" in name:
        return "MMLU-Pro"
    elif "mmlu" in name:
        return "MMLU"
    elif "openbookqa" in name:
        return "OBQA"
    elif "piqa" in name:
        return "PIQA"
    elif "socialiqa" in name:
        return "SIQA"
    elif "winogrande" in name:
        return "WinoG"
    elif "agi" in name:
        return "AGIEval"
    elif "bbh" in name:
        return "BBH"
    elif "gsm8k" in name:
        return "GSM8K"
    elif "triviaqa" in name:
        return "TriviaQA"
    elif "coqa" in name:
        return "CoQA"
    elif "drop" in name:
        return "DROP"
    elif "jeopardy" in name:
        return "JPRDY"
    elif "naturalqs" in name:
        return "NatQs"
    elif "squad" in name:
        return "SQuAD"
    else:
        return name+" TODO"
df["task"] = df["alias"].apply(rename)
df = df[~df['task'].str.contains("TODO", na=False)]
cols = df.columns[:2].tolist() + sorted(df.columns[2:])
df = df[cols]
df = df[~df["model"].str.contains("lr1e-05")]

[PosixPath('finetuning_eval/mmlu:mc::olmes/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/allenai/OLMo-2-0425-1B-SFT/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen/Qwen2.5-0.5B-Instruct/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42-merged/metrics.json'), PosixPath('finetuning_eval/mmlu:mc::olmes/meta-llama/Llama-3.2-1B-Instruct/met

In [8]:
import os
import pandas as pd


df_f = df[["task_suite", "alias", "primary_score_y", "model", "task"]].copy()
df_f = (
    df_f.groupby(["model", "task"])["primary_score_y"]
    .mean()
    .reset_index()
    .pivot(index="model", columns="task", values="primary_score_y")
)

df_f.insert(0, "Avg", df_f.mean(axis=1, numeric_only=True))

external_models = [
    "Qwen2.5-0.5B-Instruct",
    "OLMo-2-0425-1B-SFT",
    "Llama-3.2-1B-Instruct",
]

df_f = df_f.reset_index()
df_f["type"] = df_f["model"].apply(
    lambda x: "external" if x in external_models else "own"
)


score_cols = [c for c in df_f.columns if c not in ["model", "type"]]

for col in score_cols:
    max_val = df_f[col].max()

    def fmt(x):
        if pd.isna(x):
            return ""
        formatted = f"{x:.2f}".lstrip("0") if x < 1 else f"{x:.2f}"
        return f"\\textbf{{{formatted}}}" if x == max_val else formatted

    df_f[col] = df_f[col].apply(fmt)

df_f.rename(
    columns={c: f"\\rotatebox{{45}}{{{c}}}" for c in score_cols},
    inplace=True,
)

df_f["model"] = df_f["model"].apply(lambda x: x.split("_")[0])


own = df_f[df_f["type"] == "own"].drop(columns="type")
ext = df_f[df_f["type"] == "external"].drop(columns="type")

final_df = pd.concat([own, ext], axis=0)
final_df = final_df.set_index("model")
final_df.index.name = ""
final_df.columns.name = "Task"
latex = final_df.to_latex(
    escape=False,
    index=True,
    header=True,
    column_format = "l|c|" + "c" * (len(final_df.columns) - 1)
)


n_own = len(own)
n_cols = len(final_df.columns) + 1  # +1 for index column

lines = latex.splitlines()

# first midrule after header
data_start = next(
    i for i, l in enumerate(lines) if l.strip() == "\\midrule"
) + 1

insert_at = data_start + n_own
empty_cells = " & " * (n_cols - 1)

lines.insert(
    data_start,
    f"\\textbf{{Fine-tuned Models}}{empty_cells} \\\\"
)

lines.insert(
    insert_at+1,
    "\\hline\n"
    f"\\textbf{{Base Models}}{empty_cells} \\\\"
)



latex = "\n".join(lines)


os.makedirs("tables", exist_ok=True)
with open("tables/olmes-all.tex", "w") as f:
    f.write(latex)
